# SQL — Alteração e remoção de tabelas

Neste material vamos aprender a modificar e remover estruturas existentes no banco de dados utilizando:

- `ALTER TABLE`;
- `ADD COLUMN`;
- `DROP COLUMN`;
- `RENAME COLUMN`;
- `RENAME TO`;
- `ALTER COLUMN`;
- adição e remoção de constraints;
- `DROP TABLE`;
- `IF EXISTS`;
- `CASCADE`.

Esses comandos fazem parte da **DDL (Data Definition Language)**, utilizada para definir e modificar a estrutura do banco de dados.

# ALTER TABLE

Depois que uma tabela foi criada, podemos perceber que sua estrutura precisa ser modificada.

O comando `ALTER TABLE` permite realizar diversas alterações em uma tabela existente.

Sua estrutura geral é:

```sql
ALTER TABLE nome_tabela
ação;
```

Por exemplo:

```sql
ALTER TABLE clientes
ADD COLUMN telefone VARCHAR(20);
```

Nesse caso, uma nova coluna chamada `telefone` é adicionada à tabela `clientes`.

# Adicionando colunas

Podemos adicionar uma nova coluna utilizando `ADD COLUMN`.

```sql
ALTER TABLE clientes
ADD COLUMN telefone VARCHAR(20);
```

Também podemos adicionar uma coluna com restrições:

```sql
ALTER TABLE clientes
ADD COLUMN data_nascimento DATE;
```

Ou:

```sql
ALTER TABLE clientes
ADD COLUMN ativo BOOLEAN NOT NULL DEFAULT TRUE;
```

Nesse último exemplo, todos os novos registros terão `ativo` como `TRUE` por padrão.

# Adicionando múltiplas colunas

Podemos adicionar várias colunas utilizando comandos `ALTER TABLE` separados:

```sql
ALTER TABLE clientes
ADD COLUMN telefone VARCHAR(20);

ALTER TABLE clientes
ADD COLUMN data_nascimento DATE;
```

Essa abordagem também torna cada alteração explícita e fácil de identificar.

# Removendo colunas

Para remover uma coluna utilizamos `DROP COLUMN`.

```sql
ALTER TABLE clientes
DROP COLUMN telefone;
```

A coluna `telefone` será removida da tabela.

> **Atenção:** os dados armazenados nessa coluna também serão removidos.

Por isso, `DROP COLUMN` deve ser utilizado com cuidado.

# Renomeando colunas

Podemos alterar o nome de uma coluna utilizando `RENAME COLUMN`.

```sql
ALTER TABLE clientes
RENAME COLUMN nome TO nome_completo;
```

Depois dessa alteração, a coluna anteriormente chamada `nome` passa a se chamar `nome_completo`.

O conteúdo armazenado na coluna não é alterado; apenas seu nome é modificado.

# Renomeando tabelas

Também podemos alterar o nome de uma tabela:

```sql
ALTER TABLE clientes
RENAME TO clientes_ativos;
```

A tabela `clientes` passa a se chamar `clientes_ativos`.

As colunas e os registros continuam existindo.

# Alterando o tipo de uma coluna

O `ALTER TABLE` também permite alterar o tipo de uma coluna.

Por exemplo:

```sql
ALTER TABLE clientes
ALTER COLUMN telefone TYPE VARCHAR(30);
```

Nesse caso, o tamanho máximo da coluna `telefone` passa de 20 para 30 caracteres.

Outro exemplo:

```sql
ALTER TABLE produtos
ALTER COLUMN preco TYPE NUMERIC(12,2);
```

# Alterando valores padrão

Podemos adicionar um valor padrão:

```sql
ALTER TABLE produtos
ALTER COLUMN estoque SET DEFAULT 0;
```

Agora, quando um novo produto for inserido sem informar o estoque, o valor utilizado será `0`.

Também podemos remover o valor padrão:

```sql
ALTER TABLE produtos
ALTER COLUMN estoque DROP DEFAULT;
```

# Adicionando NOT NULL

Podemos adicionar a restrição `NOT NULL` posteriormente:

```sql
ALTER TABLE produtos
ALTER COLUMN nome SET NOT NULL;
```

A partir desse momento, novos registros não poderão possuir `NULL` na coluna `nome`.

É importante observar que essa alteração pode falhar caso já existam registros com `NULL` nessa coluna.

Podemos verificar antes:

```sql
SELECT *
FROM produtos
WHERE nome IS NULL;
```

# Removendo NOT NULL

Também podemos remover a obrigatoriedade:

```sql
ALTER TABLE produtos
ALTER COLUMN nome DROP NOT NULL;
```

A coluna volta a permitir valores `NULL`.

# Adicionando constraints

Também podemos adicionar uma constraint depois que a tabela já foi criada.

Por exemplo, podemos criar uma regra para impedir preços negativos:

```sql
ALTER TABLE produtos
ADD CONSTRAINT produtos_preco_check
CHECK (preco >= 0);
```

O nome `produtos_preco_check` é o nome da constraint.

Dar nomes às constraints é útil porque podemos identificá-las posteriormente.

# Removendo constraints

Para remover uma constraint utilizamos:

```sql
ALTER TABLE produtos
DROP CONSTRAINT produtos_preco_check;
```

A regra deixa de ser aplicada.

# Adicionando uma chave estrangeira

Também podemos adicionar um relacionamento depois que as tabelas já foram criadas.

Suponha que `pedidos` possua `cliente_id` e `clientes` possua `id`.

Podemos criar a relação:

```sql
ALTER TABLE pedidos
ADD CONSTRAINT pedidos_cliente_fk
FOREIGN KEY (cliente_id)
REFERENCES clientes(id);
```

Agora `pedidos.cliente_id` deve referenciar um cliente existente.

# Removendo uma chave estrangeira

Se precisarmos remover essa relação:

```sql
ALTER TABLE pedidos
DROP CONSTRAINT pedidos_cliente_fk;
```

A coluna `cliente_id` continuará existindo, mas a restrição que garantia sua referência à tabela `clientes` será removida.

# Alterações em uma tabela existente

Podemos combinar diferentes operações para evoluir uma tabela.

Por exemplo, imagine que inicialmente temos:

```sql
CREATE TABLE produtos (
    id INTEGER GENERATED ALWAYS AS IDENTITY PRIMARY KEY,
    nome VARCHAR(100) NOT NULL,
    preco NUMERIC(10,2) NOT NULL
);
```

Posteriormente decidimos que produtos também precisam possuir estoque:

```sql
ALTER TABLE produtos
ADD COLUMN estoque INTEGER NOT NULL DEFAULT 0;
```

Depois percebemos que o estoque não pode ser negativo:

```sql
ALTER TABLE produtos
ADD CONSTRAINT produtos_estoque_check
CHECK (estoque >= 0);
```

A tabela foi evoluindo sem precisar ser recriada.

# DROP TABLE

Enquanto `ALTER TABLE` modifica uma tabela existente, `DROP TABLE` remove a tabela.

```sql
DROP TABLE produtos;
```

Depois desse comando:

- a tabela deixa de existir;
- seus registros são removidos;
- sua estrutura é removida;
- suas colunas deixam de existir.

É uma operação diferente de `DELETE`.

# DELETE x DROP TABLE

É importante não confundir:

```sql
DELETE FROM produtos;
```

com:

```sql
DROP TABLE produtos;
```

O `DELETE` remove **registros**, mas mantém a tabela.

O `DROP TABLE` remove **a própria tabela**.

```text
DELETE

Banco
 └── produtos
      ├── estrutura
      └── registros ← removidos
```

Enquanto:

```text
DROP TABLE

Banco
 └── produtos ← removida
```

# DROP TABLE e dependências

No nosso modelo temos relacionamentos entre tabelas:

```text
clientes
   │
   └── pedidos
          │
          └── itens_pedido
```

Por exemplo, `pedidos` possui uma chave estrangeira que referencia `clientes`.

Se tentarmos remover `clientes` enquanto `pedidos` ainda depende dela:

```sql
DROP TABLE clientes;
```

o PostgreSQL poderá impedir a operação por causa da dependência.

Precisamos primeiro remover a tabela dependente:

```sql
DROP TABLE pedidos;

DROP TABLE clientes;
```

Em um banco com várias relações, é importante conhecer essas dependências antes de remover estruturas.

# DROP TABLE CASCADE

O PostgreSQL permite utilizar `CASCADE`:

```sql
DROP TABLE clientes CASCADE;
```

Nesse caso, o PostgreSQL também remove os objetos que dependem da tabela.

Isso pode ser conveniente durante um laboratório ou na reconstrução de uma estrutura, mas deve ser utilizado com cuidado.

Uma única instrução pode provocar a remoção de diversos objetos relacionados.

# IF EXISTS

Podemos utilizar `IF EXISTS` para evitar erro caso uma tabela não exista.

```sql
DROP TABLE IF EXISTS produtos;
```

Se `produtos` existir, ela será removida.

Se não existir, o PostgreSQL não apresentará um erro de tabela inexistente.

Isso é bastante útil em scripts que podem ser executados várias vezes.

# IF EXISTS com ALTER TABLE

O mesmo conceito pode ser utilizado com `ALTER TABLE`:

```sql
ALTER TABLE IF EXISTS produtos
ADD COLUMN estoque INTEGER;
```

Se a tabela não existir, a alteração simplesmente não será realizada.

# Exemplo de evolução de uma tabela

Podemos simular a evolução de uma tabela ao longo do tempo.

Começamos com:

```sql
CREATE TABLE alunos (
    id INTEGER GENERATED ALWAYS AS IDENTITY PRIMARY KEY,
    nome VARCHAR(100) NOT NULL
);
```

Depois percebemos que precisamos armazenar o e-mail:

```sql
ALTER TABLE alunos
ADD COLUMN email VARCHAR(150);
```

Depois decidimos que o e-mail deve ser único:

```sql
ALTER TABLE alunos
ADD CONSTRAINT alunos_email_unique
UNIQUE (email);
```

Também queremos armazenar a data de nascimento:

```sql
ALTER TABLE alunos
ADD COLUMN data_nascimento DATE;
```

E finalmente percebemos que o nome da coluna deveria ser `nome_completo`:

```sql
ALTER TABLE alunos
RENAME COLUMN nome TO nome_completo;
```

A estrutura final foi construída gradualmente através de alterações.

# CREATE, ALTER e DROP

Os comandos estudados neste material fazem parte da **DDL — Data Definition Language**.

| Comando | Finalidade |
|---|---|
| `CREATE TABLE` | cria uma tabela |
| `ALTER TABLE` | modifica uma tabela |
| `DROP TABLE` | remove uma tabela |

Podemos pensar no ciclo:

```text
CREATE
   ↓
ALTER
   ↓
ALTER
   ↓
ALTER
   ↓
DROP
```

`CREATE` cria a estrutura, `ALTER` permite evoluí-la e `DROP` remove a estrutura.